# Qinfer算子推理

In [ ]:
import pandas as pd
from transformers import pipeline
import torch
from tqdm import tqdm
import os

# 设置模型文件路径
datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
model_dir = '/home/wangshuo/resource/AIModels/NLP/bart-large-mnli'

# 确保模型运行在 GPU 上（如果 GPU 可用）
device = 0 if torch.cuda.is_available() else -1  # 使用 GPU (0) 或者 CPU (-1)

# 加载文本推断 pipeline，指定使用 GPU（device=0）或 CPU（device=-1）
nli_pipeline = pipeline("text-classification", model=model_dir, tokenizer=model_dir, device=device)

def check_nli(topic, post_body):
    """
    使用文本推断模型判断 post 是否符合 topic
    """
    hypothesis = f"This text is about {topic}."  # 主题假设
    result = nli_pipeline(f"{post_body} [SEP] {hypothesis}")
    return result

# 读取CSV文件
# csv_file = '/home/wangshuo/resource/datasets/workload_20w/posts_trimmed1.csv'
csv_file = datadir + '/post.csv'
df = pd.read_csv(csv_file)

# 主题定义
# topic = "Trump will make america great again"  # 设置主题
topic = "I support Trump"  # 设置主题

# 为每个 post 执行 NLI 推理
nli_labels = []
probabilities = []

# 使用 tqdm 显示进度条
for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing Posts"):
    post_body = row['body']  # 获取每个 post 的 body 内容

    if not post_body:  # 如果 body 为空，跳过
        nli_labels.append(None)
        probabilities.append(None)
        continue

    try:
        # 使用 NLI 模型进行推理
        result = check_nli(topic, post_body)
        
        # 获取推理结果的标签和概率
        label = result[0]['label']
        probability = result[0]['score']
        
        # 存储结果
        nli_labels.append(label)
        probabilities.append(probability)
        
    except Exception as e:
        print(f"Error processing post ID {row['id']}: {e}")
        nli_labels.append(None)
        probabilities.append(None)

# 将结果添加到 DataFrame 中
df['nli_label'] = nli_labels
df['probability'] = probabilities

# 保存包含新列的 CSV 文件
output_csv_file = '/home/wangshuo/resource/datasets/workload_20w/posts_with_nli_results_test.csv'
df.to_csv(output_csv_file, index=False)

print(f"推理结果已保存到 {output_csv_file}")


In [ ]:
import pandas as pd
from transformers import pipeline
import torch
from tqdm import tqdm
from datasets import Dataset
import os

# 设置模型文件路径
model_dir = '/home/wangshuo/resource/AIModels/NLP/bart-large-mnli'
datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
# 确保模型运行在 GPU 上（如果 GPU 可用）
device = 0 if torch.cuda.is_available() else -1  # 使用 GPU (0) 或者 CPU (-1)

# 加载文本推断 pipeline，指定使用 GPU（device=0）或 CPU（device=-1）
nli_pipeline = pipeline("text-classification", model=model_dir, tokenizer=model_dir, device=device)

def check_nli_batch(topic, post_bodies_batch):
    """
    使用文本推断模型批量判断 posts 是否符合 topic
    """
    # 构建每个 post 和主题的假设
    hypotheses = [f"This text is about {topic}." for _ in post_bodies_batch]
    
    # 合并 posts 和 hypotheses
    inputs = [f"{body} [SEP] {hypothesis}" for body, hypothesis in zip(post_bodies_batch, hypotheses)]
    
    # 批量推理
    results = nli_pipeline(inputs)
    return results

# 读取CSV文件
# csv_file = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_2/trump_posts_filtered.csv'
csv_file = datadir + '/post.csv'
df = pd.read_csv(csv_file)

# 主题定义
# topic = "Trump will make america great again"  # 设置主题
topic = "I support Trump"  # 设置主题

# 将 pandas DataFrame 转换为 datasets.Dataset 格式
dataset = Dataset.from_pandas(df[['id:ID', 'body']])

# 设置批处理大小
batch_size = 128  # 你可以根据显存大小调整

# 存储推理结果
nli_labels = []
probabilities = []

# 使用 tqdm 显示进度条
for i in tqdm(range(0, len(dataset), batch_size), total=len(dataset)//batch_size, desc="Processing Posts in Batches"):
    batch = dataset[i:i + batch_size]
    post_bodies_batch = batch['body']  # 获取这一批次的所有 post 的 body 内容
    # 过滤空的 post body
    post_bodies_batch = [body for body in post_bodies_batch if body]

    if len(post_bodies_batch) == 0:  # 如果这一批次都为空，跳过
        continue

    try:
        # 使用批量 NLI 模型进行推理
        results = check_nli_batch(topic, post_bodies_batch)
        
        # 提取推理结果
        for result in results:
            label = result['label']
            probability = result['score']
            nli_labels.append(label)
            probabilities.append(probability)
        
    except Exception as e:
        print(f"Error processing batch {i//batch_size}: {e}")
        nli_labels.extend([None] * len(post_bodies_batch))  # 出现错误时，填充 None
        probabilities.extend([None] * len(post_bodies_batch))

# 将结果添加到 DataFrame 中
df['ML1_label'] = nli_labels
df['probability'] = probabilities

# 保存包含新列的 CSV 文件
# output_csv_file = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_2/posts_with_nli_results_test.csv'
output_csv_file = datadir+ '/post_ML1.csv'
df.to_csv(output_csv_file, index=False)

print(f"推理结果已保存到 {output_csv_file}")


#### 1.使用bart-large-mnli对comment/post进行文本推理

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm  # 导入 tqdm

# 设置模型文件路径
model_dir = '/home/wangshuo/resource/AIModels/NLP/NLI/bart-large-mnli'

# 加载模型和 tokenizer（从本地加载）
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

# 将模型转移到GPU（如果可用）
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.half()
model.to(device)
print(device)

# 读取 CSV 文件中的数据
# data_dir = r'/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3'
datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
csv_file = datadir + '/post1.csv'
print(csv_file)
df = pd.read_csv(csv_file)

# 获取所有的 "body" 字段内容
all_posts = df['body'].tolist()
print(f"数据加载完毕，共有 {len(all_posts)} 个帖子")

# 定义推理函数
def check_nli_batch(topic, batch_posts):
    """
    对每一批次的 post 进行 NLI 推理，判断其是否与给定的 topic 匹配
    """
    # 创建每个输入样本的 "topic + post" 对
    batch_inputs = [f"{post} [SEP] {topic}" for post in batch_posts]

    # 编码批量输入
    encoding = tokenizer(batch_inputs, padding=True, truncation=True, return_tensors="pt")
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # 模型推理
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # 选择 "entailment" 和 "contradiction" 的 logits
        entail_contradiction_logits = logits[:, [0, 2]]

        # 计算 softmax 概率
        probs = entail_contradiction_logits.softmax(dim=1)

        # 获取 "entailment" 类别的概率（标签为真时的概率）
        prob_label_is_true = probs[:, 1].cpu().numpy()
        
        return prob_label_is_true

# 假设每批次处理 32 个文本
batch_size = 32
results = []

# topic = "Trump will win the presidency"
# topic = "Trump will make america great again"
# topic = "Trump will win and save america"  # workload_1

topic = "I support Trump"  # workload_1
# 使用 tqdm 显示进度条
for i in tqdm(range(0, len(all_posts), batch_size), desc="Processing Posts"):
    batch_posts = all_posts[i:i + batch_size]
    probabilities = check_nli_batch(topic, batch_posts)
    # 将每个样本的 "entailment" 概率添加到结果中
    results.extend(probabilities)

# 将结果添加到 DataFrame
df['ML1_probability'] = results
# 根据概率判断标签是否为 "entailment"（阈值为 0.5）
df['ML1_label'] = df['ML1_probability'].apply(lambda x: "ML1_entailment" if x and x > 0.3 else "ML1_not")

# 保存包含新列的 CSV 文件
output_csv_file = datadir + '/post_ML.csv'
df.to_csv(output_csv_file, index=False)

print(f"推理结果已保存到 {output_csv_file}")


不拼接和单个测试相同的批处理NLI方式

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm

# ————————————————
# 1. 配置部分
# ————————————————
model_dir = '/home/wangshuo/resource/AIModels/NLP/bart-large-mnli'
data_dir  = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
input_csv = data_dir + '/post1.csv'
output_csv= data_dir + '/post_ML.csv'

THRESHOLD  = 0.3
BATCH_SIZE = 32
topic      = "I support Trump"

# ————————————————
# 2. 加载模型和 tokenizer
# ————————————————
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model     = AutoModelForSequenceClassification.from_pretrained(model_dir)
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ————————————————
# 3. 定义批量推理函数，使用 “This text is about {topic}.”
# ————————————————
def check_nli_batch(topic, posts):
    """
    对一批 posts 做 NLI 推理，假设句为 "This text is about {topic}."
    返回：每条 post 的 entailment 概率（numpy array）
    """
    # 构造 hypothesis 列表
    hypotheses = [f"This text is about {topic}." for _ in posts]

    # 使用 text_pair，将 posts 作为 premise、hypotheses 作为 hypothesis
    encoding = tokenizer(
        posts,
        hypotheses,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**encoding).logits

    # 只取 contradiction(0) 和 entailment(2) 的 logits
    ec_probs = logits[:, [0, 2]].softmax(dim=1)
    # 返回 entailment（index=1）的概率
    return ec_probs[:, 1].cpu().numpy()

# ————————————————
# 4. 读取数据并执行批量推理
# ————————————————
df = pd.read_csv(input_csv)
posts = df['body'].fillna("").astype(str).tolist()

results = []
for i in tqdm(range(0, len(posts), BATCH_SIZE), desc="Processing Posts"):
    batch = posts[i:i+BATCH_SIZE]
    probs = check_nli_batch(topic, batch)
    results.extend(probs)

# ————————————————
# 5. 保存结果到 CSV
# ————————————————
df['ML1_probability'] = results
df['ML1_label'] = df['ML1_probability'].apply(
    lambda p: "ML1_entailment" if p > THRESHOLD else "ML1_not"
)
df.to_csv(output_csv, index=False)
print(f"推理结果已保存到 {output_csv}")


## bart ： 12W comment 15min
## distilbart： 12w comment 11:15 min 

In [ ]:
import pandas as pd
import numpy as np

# 读取原始数据 CSV 文件
csv_file = '/home/wangshuo/resource/datasets/workload_20w/comments_with_nli_results.csv'
df = pd.read_csv(csv_file)

# 假设你已经得到了处理后的结果列表 results，长度为 126193
# 假设 'results' 已经包含了有效的 "entailment" 概率

# 创建一个新的列表，用于保存结果（包括 NaN 或 None 的行）
nli_results = []

# 获取 'body' 字段内容，并确保处理顺序与原始数据匹配
all_posts = df['body'].tolist()

# 变量 results 是非 NaN 或 None 的有效结果
valid_results_iter = iter(results)  # 使得结果可以迭代

# 遍历原始的 'body' 字段数据
for post in all_posts:
    if isinstance(post, str) and post.strip():  # 仅当 post 不是 NaN 或 None
        nli_results.append(next(valid_results_iter))  # 使用有效的推理结果
    else:
        nli_results.append(None)  # 对于 NaN 或 None 的行，保留 None 或 NaN

# 将推理结果添加到 DataFrame 中
df['bart_entailment_probability'] = nli_results

# 根据概率判断标签是否为 "entailment"（阈值为 0.5）
df['nli_label'] = df['bart_entailment_probability'].apply(lambda x: "entailment" if x and x > 0.3 else "not")

# 保存包含新列的 CSV 文件
output_csv_file = '/home/wangshuo/resource/datasets/workload_20w/comments_with_nli_results.csv'
df.to_csv(output_csv_file, index=False)

print(f"推理结果已保存到 {output_csv_file}")


#### 1.1.使用distilbart-mnli-12-6对comments进行推理

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm

# ————————————————
# 1. 配置部分
# ————————————————
model_dir = '/home/wangshuo/resource/AIModels/NLP/distilbart-mnli-12-6'
datadir   = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
input_csv = datadir + '/post_LLM_cleaned_6.csv'
output_csv= datadir + '/post_LLM_cleaned_6.csv'

THRESHOLD  = 0.5
BATCH_SIZE = 32
topic      = "I support Trump"

# ————————————————
# 2. 加载模型和 tokenizer
# ————————————————
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model     = AutoModelForSequenceClassification.from_pretrained(model_dir)
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.half()
model.to(device)
model.eval()

# ————————————————
# 3. 批量推理函数，使用 “This text is about {topic}.”
# ————————————————
def check_nli_batch(topic, posts):
    """
    对一批 posts 做 NLI 推理，假设句为 "This text is about {topic}."
    返回：每条 post 的 entailment 概率（numpy array）
    """
    # 构造 hypothesis 列表
    hypotheses = [f"This text is about {topic}." for _ in posts]

    # 使用 text_pair，将 posts 作为 premise、hypotheses 作为 hypothesis
    encoding = tokenizer(
        posts,
        hypotheses,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**encoding).logits

    # 只取 contradiction(0) 和 entailment(2) 的 logits
    ec_probs = logits[:, [0, 2]].softmax(dim=1)
    # 返回 entailment（index=1）的概率
    return ec_probs[:, 1].cpu().numpy()

# ————————————————
# 4. 读取数据并执行批量推理
# ————————————————
df = pd.read_csv(input_csv)
posts = df['body'].fillna("").astype(str).tolist()

results = []
for i in tqdm(range(0, len(posts), BATCH_SIZE), desc="Processing Posts"):
    batch = posts[i:i+BATCH_SIZE]
    probs = check_nli_batch(topic, batch)
    results.extend(probs)

# ————————————————
# 5. 保存结果到 CSV
# ————————————————
df['ML1_proxy1_probability'] = results
df['ML1_proxy1_label'] = df['ML1_proxy1_probability'].apply(
    lambda p: "ML1_proxy1_entailment" if p > THRESHOLD else "ML1_proxy1_not"
)
df.to_csv(output_csv, index=False)
print(f"推理结果已保存到 {output_csv}")


#### 1.2使用 DeepSpeed-Inference 和fp16半精度  加速distilbart-mnli-12-6对comments的推理 
速度比bart-mnli-12-6 快5倍

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import deepspeed
from tqdm import tqdm

# —— 1. 模型路径 & 设备 ——  
model_dir = '/home/wangshuo/resource/AIModels/NLP/distilbart-mnli-12-6'
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# —— 2. 加载 tokenizer 和原始模型 ——  
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model     = AutoModelForSequenceClassification.from_pretrained(model_dir)

# —— 3. 半精度（FP16）并转到 GPU ——  
model.half()
model.to(device)
model.eval()

# —— 4. DeepSpeed-Inference 注入 ——  
#    安装 bitsandbytes / deepspeed 后使用：
model = deepspeed.init_inference(
    model,                                  # 原始模型
    mp_size=1,                              # 单卡推理
    # dtype=torch.float16,                    # 推理时用半精度
    replace_method="auto",                  # 自动替换支持的层
    replace_with_kernel_inject=True         # 注入 DeepSpeed-kernels
)

# —— 5. 读取数据 ——  
datadir  = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
in_csv   = f"{datadir}/post_ML1.csv"
df       = pd.read_csv(in_csv)
all_posts = df['body'].fillna("").astype(str).tolist()
print(f"加载完毕，共 {len(all_posts)} 条帖子")

# —— 6. 定义推理函数 ——  
def check_nli_batch(topic, batch_posts):
    texts = [f"{post} [SEP] {topic}" for post in batch_posts]
    enc = tokenizer(texts, padding=True, truncation=True,
                    return_tensors="pt", max_length=256).to(device)
    with torch.no_grad():
        outputs = model(**enc)
        logits = outputs.logits
        # 只取 “entailment vs contradiction”
        entail_contra = logits[:, [0, 2]]
        probs = entail_contra.softmax(dim=1)
        return probs[:, 1].cpu().numpy()

# —— 7. 批量推理 ——  
batch_size = 32
results = []
for i in tqdm(range(0, len(all_posts), batch_size), desc="DeepSpeed 推理"):
    batch = all_posts[i:i+batch_size]
    probs = check_nli_batch("I support Trump", batch)
    results.extend(probs)

# —— 8. 保存结果 ——  
df['ML1_proxy1_probability'] = results
df['ML1_proxy1_label'] = df['ML1_proxy1_probability'].apply(
    lambda x: "ML1_proxy1_entailment" if x > 0.5 else "ML1_proxy1_not"
)
out_csv = f"{datadir}/post_ML1_deepspeed.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"推理结果已保存到：{out_csv}")


#### 1.4 Alireza1044/albert-base-v2-mnli  用albert-base-v2-MNLI完成自然语言推理任务

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import deepspeed
from tqdm import tqdm

# —— 1. 模型 ID & 设备 ——  
model_id = "/home/wangshuo/resource/AIModels/NLP/NLI/albert-base-v2-mnli"
device   = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# —— 2. 加载 tokenizer 和原始模型 ——  
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    torch_dtype=torch.float16,       # 直接以半精度加载
    device_map="auto"                # 自动映射到可用 GPU
)

# —— 3. 切换到 eval 模式 ——  
model.eval()

# —— 4. DeepSpeed‑Inference 注入 ——  
#    注意新版 DeepSpeed 建议使用 tensor_parallel 而非 mp_size，并且不必重复指定 dtype
ds_engine = deepspeed.init_inference(
    model,                                  
    tensor_parallel={"tp_size": 1},         # 单卡推理：tp_size=1
    replace_with_kernel_inject=True,        # 注入 DeepSpeed‑kernels
    inference_device="cuda:1" if torch.cuda.is_available() else "cpu"
)
model = ds_engine.module  # 拿回包装后的模型

# —— 5. 读取数据 ——  
datadir  = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
in_csv   = f"{datadir}/post_ML1.csv"
df       = pd.read_csv(in_csv)
all_posts = df['body'].fillna("").astype(str).tolist()
print(f"加载完毕，共 {len(all_posts)} 条帖子")

# —— 6. 定义推理函数 ——  
def check_nli_batch(topic, batch_posts):
    # ALBERT-base‑v2‑MNLI 模型默认标签顺序 [contradiction, neutral, entailment]
    texts = [f"{post} [SEP] {topic}" for post in batch_posts]
    enc = tokenizer(
        texts, padding=True, truncation=True,
        return_tensors="pt", max_length=256
    ).to(model.device)
    with torch.no_grad():
        outputs = model(**enc)
        logits = outputs.logits
        # 取 “entailment vs contradiction”
        entail_contra = logits[:, [0, 2]]
        probs = entail_contra.softmax(dim=1)
        return probs[:, 1].cpu().numpy()

# —— 7. 批量推理 ——  
batch_size = 32
results = []
for i in tqdm(range(0, len(all_posts), batch_size), desc="DeepSpeed 推理"):
    batch = all_posts[i : i + batch_size]
    probs = check_nli_batch("I support Trump", batch)
    results.extend(probs)

# —— 8. 保存结果 ——  
df['ML1_proxy2_probability'] = results
df['ML1_proxy2_label'] = df['ML1_proxy2_probability'].apply(
    lambda x: "ML1_proxy2_entailment" if x > 0.8 else "ML1_proxy2_not"
)
out_csv = f"{datadir}/post_ML1_albert_mnli.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"推理结果已保存到：{out_csv}")


M-FAC/bert-mini-finetuned-mnli 

In [6]:
# #### 0.使用deberta-v2-xlarge-mnli对comment/post进行文本推理
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm


# ————————————————
# 1. 配置部分
# ————————————————
# model_dir = '/home/wangshuo/resource/AIModels/NLP/NLI/albert-base-v2-mnli'
model_dir = '/home/wangshuo/resource/AIModels/Finetune/NLI/proxy_distilbert-base-uncased-finetuned-mnli_epoch10'
data_dir  = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
input_csv = data_dir + '/post_origin.csv'
output_csv= data_dir + '/post_origin.csv'

THRESHOLD  = 0.3
BATCH_SIZE = 32
topic      = "I support Trump"

# ————————————————
# 2. 加载模型和 tokenizer
# ————————————————
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model     = AutoModelForSequenceClassification.from_pretrained(model_dir)
device    = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
model.to(device)
# model.half()
model.eval()

# ————————————————
# 3. 定义批量推理函数，使用 “This text is about {topic}.”
# ————————————————
def check_nli_batch(topic, posts):
    """
    对一批 posts 做 NLI 推理，假设句为 "This text is about {topic}."
    返回：每条 post 的 entailment 概率（numpy array）
    """
    # 构造 hypothesis 列表
    hypotheses = [f"This text is about {topic}." for _ in posts]

    # 使用 text_pair，将 posts 作为 premise、hypotheses 作为 hypothesis
    encoding = tokenizer(
        posts,
        hypotheses,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**encoding).logits

    # 只取 contradiction(0) 和 entailment(2) 的 logits
    ec_probs = logits[:, [0, 2]].softmax(dim=1)
    # 返回 entailment（index=1）的概率
    return ec_probs[:, 1].cpu().numpy()

# ————————————————
# 4. 读取数据并执行批量推理
# ————————————————
df = pd.read_csv(input_csv)
posts = df['body'].fillna("").astype(str).tolist()

results = []
for i in tqdm(range(0, len(posts), BATCH_SIZE), desc="Processing Posts"):
    batch = posts[i:i+BATCH_SIZE]
    probs = check_nli_batch(topic, batch)
    results.extend(probs)

# ————————————————
# 5. 保存结果到 CSV
# ————————————————
df['ML1_proxy3f_probability'] = results
df.to_csv(output_csv, index=False)
print(f"推理结果已保存到 {output_csv}")


Processing Posts: 100%|██████████| 769/769 [00:39<00:00, 19.26it/s]


推理结果已保存到 /home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/post_origin.csv


## 2.1使用cardiffnlp/twitter-roberta-base-sentiment对comment/post进行情感分析

2.1.1 测试用例

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax

# 1. 指定模型
MODEL = "/home/wangshuo/resource/AIModels/NLP/TE/twitter-roberta-base-sentiment"

# 2. 加载 tokenizer 和 model
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL)
device    = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 3. 下载并构造标签映射
import csv
mapping_file = "/home/wangshuo/resource/AIModels/NLP/TE/twitter-roberta-base-sentiment/mapping.txt"  # 或者写成绝对路径 "/home/wangshuo/.../mapping.txt"

labels = []
with open(mapping_file, encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    for row in reader:
        if len(row) >= 2:
            labels.append(row[1])

# 4. 预处理函数
def preprocess(text: str) -> str:
    tokens = []
    for t in text.split():
        if t.startswith("@") and len(t) > 1:
            tokens.append("@user")
        elif t.startswith("http"):
            tokens.append("http")
        else:
            tokens.append(t)
    return " ".join(tokens)

# 5. 推理函数
def sentiment_batch(texts):
    texts = [preprocess(t) for t in texts]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        outputs = model(**enc)
        scores = outputs.logits.cpu().numpy()
    probs   = softmax(scores, axis=1)
    idxs    = probs.argmax(axis=1)
    labs    = [labels[i] for i in idxs]
    vals    = probs.max(axis=1)
    return list(zip(labs, vals))

# 6. 测试用例
if __name__ == "__main__":
    samples = [
        "I absolutely love this product! It works like a charm.",
        "This is the worst experience I've ever had.",
        "Meh, it was okay—not great but not terrible either.",
        "Wow, fantastic service and very friendly staff!",
        "I'm not impressed. Expected much better."
    ]
    results = sentiment_batch(samples)
    for text, (lab, score) in zip(samples, results):
        print(f"Text: {text}\n  → Sentiment: {lab:<8} (score: {score:.4f})\n")


In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from scipy.special import softmax
from tqdm import tqdm

# 1. 模型与 tokenizer 路径或名称
MODEL = "/home/wangshuo/resource/AIModels/NLP/TE/TweetEval_DistilBERT_5E"

# 2. 加载 tokenizer 和模型
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

# 3. 下载并读取标签映射（0: negative, 1: neutral, 2: positive）
#    如果不希望运行时下载，也可以手动提前下载 mapping.txt 并改为本地路径
import csv
mapping_file = "/home/wangshuo/resource/AIModels/NLP/TE/twitter-roberta-base-sentiment/mapping.txt"  # 或者写成绝对路径 "/home/wangshuo/.../mapping.txt"
labels = []
with open(mapping_file, encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    for row in reader:
        if len(row) >= 2:
            labels.append(row[1])

# 4. 定义预处理函数（将 @user、链接 等替换为占位符）
def preprocess(text: str) -> str:
    tokens = []
    for t in text.split():
        if t.startswith("@") and len(t) > 1:
            tokens.append("@user")
        elif t.startswith("http"):
            tokens.append("http")
        else:
            tokens.append(t)
    return " ".join(tokens)

# 5. 读取你的 CSV，假设有一列叫 'body'
datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
in_csv = f"{datadir}/comment_test.csv"    # 替换为你的文件名
out_csv = f"{datadir}/comment_test.csv"
df = pd.read_csv(in_csv)
all_texts = df['body'].fillna("").astype(str).tolist()

# 6. 批量推理函数
def sentiment_batch(batch_texts):
    # 6.1 预处理
    texts = [preprocess(t) for t in batch_texts]
    # 6.2 编码
    enc = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=128)
    input_ids = enc.input_ids.to(device)
    attention_mask = enc.attention_mask.to(device)
    # 6.3 推理
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        scores = outputs.logits.cpu().numpy()
    # 6.4 软最大化
    probs = softmax(scores, axis=1)
    # 6.5 取最高概率及对应标签
    max_ids = probs.argmax(axis=1)
    max_probs = probs.max(axis=1)
    labels_out = [labels[i] for i in max_ids]
    return labels_out, max_probs

# 7. 循环处理并记录结果
batch_size = 64
sent_labels = []
sent_scores = []

for i in tqdm(range(0, len(all_texts), batch_size), desc="Sentiment Analysis"):
    batch = all_texts[i:i+batch_size]
    lbs, scs = sentiment_batch(batch)
    sent_labels.extend(lbs)
    sent_scores.extend(scs)

# 8. 写回 DataFrame 并保存
df['ML2_oracle2_label'] = sent_labels
df['ML2_oracle2_probability'] = sent_scores
df.to_csv(out_csv, index=False)
print(f"情感分析结果已保存至：{out_csv}")


## 2.2使用bert-mini-finetuned-sst2对comment/post进行情感分析

2.2.1 测试用例

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax

# —— 1. 准备模型 ——  
MODEL_PATH = "/home/wangshuo/resource/AIModels/NLP/TE/bert-mini-finetuned-sst2"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_PATH)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(
    torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
).eval()

# —— 2. 手动标签映射 ——  
id2label = {0: "NEGATIVE", 1: "POSITIVE"}

# —— 3. 推理函数 ——  
def infer_batch(batch_texts):
    enc = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits.cpu().numpy()
    probs = softmax(logits, axis=1)
    pred_ids    = probs.argmax(axis=1)
    pred_scores = probs.max(axis=1)
    pred_labels = [id2label[i] for i in pred_ids]
    return list(zip(pred_labels, pred_scores))

# —— 4. 测试用例 ——  
test_comments = [
    "I absolutely love this product! It works like a charm.",
    "This is the worst experience I've ever had.",
    "Meh, it was okay—not great but not terrible either.",
    "Wow, fantastic service and very friendly staff!",
    "I'm not impressed. Expected much better."
]

results = infer_batch(test_comments)

# —— 5. 打印结果 ——  
for text, (label, score) in zip(test_comments, results):
    print(f"Comment: {text!r}")
    print(f"→ Prediction: {label}, Confidence: {score:.4f}\n")


In [ ]:
import pandas as pd
import torch
from transformers import pipeline, AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 1. 配置
model_dir = '/home/wangshuo/resource/AIModels/Finetune/distil-proxy/electra_entailment_final'
data_dir  = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
input_csv = f'{data_dir}/post_LLM_cleaned_6.csv'
output_csv= f'{data_dir}/post_LLM_cleaned_6.csv'

# 2. 构建 pipeline
device = 0 if torch.cuda.is_available() else -1
nlp_pipe = pipeline(
    "text-classification",
    model=AutoModelForSequenceClassification.from_pretrained(
        model_dir,
        config=AutoConfig.from_pretrained(model_dir, num_labels=2),
        ignore_mismatched_sizes=True
    ),
    tokenizer=AutoTokenizer.from_pretrained(model_dir),
    device=device,
    return_all_scores=True
)

# 3. 逐条推理
df = pd.read_csv(input_csv)
probs = []

for body in tqdm(df['body'].fillna(""), desc="Proxy6 Inference"):
    hypothesis = "This text is about I support Trump."
    # 正确调用：第一个位置参数是 text，hypothesis 必须用 text_pair= 指定
    output = nlp_pipe(body, text_pair=hypothesis)
    # output 格式： [[{'label':'LABEL_0','score':...}, {'label':'LABEL_1','score':...}]]
    scores = output[0]
    p_entail = next(item['score'] for item in scores if item['label'].endswith("1"))
    probs.append(p_entail)

# 4. 写回并保存
df['ML1_proxy6_probability'] = probs
df.to_csv(output_csv, index=False)
print(f"结果已保存到 {output_csv}")


In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm

# ————————————————
# 1. 配置部分
# ————————————————
model_name = 'proxy_bert-mini-finetuned-mnli_epoch20'
proxy_model_dir_pre = '/home/wangshuo/resource/AIModels/Finetune/NLI/'
model_dir_pre = '/home/wangshuo/resource/AIModels/NLP/NLI/'

model_dir = proxy_model_dir_pre + model_name
data_dir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
input_csv = data_dir + '/post_origin.csv'
output_csv = data_dir + '/post_origin.csv'

THRESHOLD = 0.3
BATCH_SIZE = 32
topic = "I support Trump"

# ————————————————
# 2. 加载模型和 tokenizer
# ————————————————
print(f'loading {model_name}')
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.half()
model.to(device)


# ————————————————
# 3. 定义批量推理函数，使用 “This text is about {topic}.”
# ————————————————
def check_nli_batch(topic, posts):
    """
    对一批 posts 做 NLI 推理，假设句为 "This text is about {topic}."
    返回：每条 post 的 entailment 概率（numpy array）
    """
    # 构造 hypothesis 列表
    hypotheses = [f"This text is about {topic}." for _ in posts]

    # 使用 text_pair，将 posts 作为 premise、hypotheses 作为 hypothesis
    encoding = tokenizer(
        posts,
        hypotheses,
        padding=True,
        truncation=True,
        return_tensors="pt",
        max_length=512
    ).to(device)

    with torch.no_grad():
        logits = model(**encoding).logits

    # 只取 contradiction(0) 和 entailment(2) 的 logits
    ec_probs = logits[:, [0, 2]].softmax(dim=1)
    # 返回 entailment（index=1）的概率
    return ec_probs[:, 1].cpu().numpy()


# ————————————————
# 4. 读取数据并执行批量推理
# ————————————————
df = pd.read_csv(input_csv)
posts = df['body'].fillna("").astype(str).tolist()

results = []
for i in tqdm(range(0, len(posts), BATCH_SIZE), desc="Processing Posts"):
    batch = posts[i:i + BATCH_SIZE]
    probs = check_nli_batch(topic, batch)
    results.extend(probs)

# ————————————————
# 5. 保存结果到 CSV
# ————————————————
df['ML1_proxy2f_probability'] = results
df.to_csv(output_csv, index=False)
print(f"推理结果已保存到 {output_csv}")
